In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, lower, regexp_extract, length

spark = (
    SparkSession.builder.appName("Analyzing the vocabulary of Pride and Prejudice.")
    .config("spark.sql.repl.eagerEval.enabled", "True")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("OFF")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 16:29:47 WARN Utils: Your hostname, humacao, resolves to a loopback address: 127.0.1.1; using 192.168.68.83 instead (on interface wlp2s0)
26/03/08 16:29:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 16:29:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
book = spark.read.text("./data/gutenberg_books/1342-0.txt")

In [3]:
book.printSchema()

root
 |-- value: string (nullable = true)



In [5]:
book.show(10, truncate=False)

+--------------------------------------------------------------------+
|value                                                               |
+--------------------------------------------------------------------+
|The Project Gutenberg EBook of Pride and Prejudice, by Jane Austen  |
|                                                                    |
|This eBook is for the use of anyone anywhere at no cost and with    |
|almost no restrictions whatsoever.  You may copy it, give it away or|
|re-use it under the terms of the Project Gutenberg License included |
|with this eBook or online at www.gutenberg.org                      |
|                                                                    |
|                                                                    |
|Title: Pride and Prejudice                                          |
|                                                                    |
+--------------------------------------------------------------------+
only s

In [6]:
lines = book.select(split(col("value"), " ").alias("line"))

words = lines.select(explode(col("line")).alias("word"))

words_lower = words.select(lower(col("word")).alias("word_lower"))

words_clean = words_lower.select(
    regexp_extract(col("word_lower"), "[a-z]+", 0).alias("word")
)

words_nonull = words_clean.filter(col("word") != "")

results = (
    words_nonull.select(length(col("word")).alias("word_length"))
    .groupby("word_length")
    .count()
)

# groups = words_nonull.groupBy(col("word"))
# results = groups.count()
results.show()

results.write.mode("overwrite").csv("./data/simple_count.csv")
results.coalesce(1).write.mode("overwrite").csv(
    "./data/simple_count_single_partition.csv"
)

+-----------+-----+
|word_length|count|
+-----------+-----+
|         12|  812|
|          1| 4116|
|         13|  393|
|          6| 9276|
|         16|    5|
|          3|28831|
|          5|11998|
|         15|   32|
|          9| 5165|
|         17|    3|
|          4|22213|
|          8| 5121|
|          7| 8679|
|         10| 2455|
|         11| 1386|
|         14|  107|
|          2|23856|
+-----------+-----+

